# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya – Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze a Croissant-defined dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library. The steps include metadata inspection, record set and field discovery (all via `@id`), loading data, basic preprocessing, and visualization.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Make sure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the Croissant package and inspect dataset metadata. We use the croissant schema URL for reproducibility.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset package
dataset = mlc.Dataset(croissant_url)

# Access and print key metadata (treat as a single object, not a dict)
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {getattr(dataset.metadata, 'version', None)}")
print(f"License: {getattr(dataset.metadata, 'license', None)}")


## 2. Data Overview
### List Record Sets and Fields (`@id`)
Explore all record sets in the dataset, each referenced by its unique `@id`. For each record set, display its fields (columns) with their `@id`s and names.


In [ ]:
# List all record sets with their @id and each field @id
print("Record sets available in the dataset:")
record_set_ids = []
for rs in dataset.list_record_sets():
    print(f"\n- RecordSet @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    fields = dataset.list_fields(rs['@id'])
    if fields:
        for field in fields:
            fname = field.get('name', '-')
            print(f"    • Field @id: {field['@id']} | name: {fname}")
    else:
        print("    (No fields listed)")

print(f"\nAll record set @ids: {record_set_ids}")

## 3. Data Extraction
Load the records for each record set (by `@id`) into Pandas DataFrames for downstream analysis. Each DataFrame key in the returned dictionary is the record set's `@id`.


In [ ]:
# Load all record sets into separate DataFrames using their @id
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Columns in DataFrame: {dataframes[rs_id].columns.tolist()}")
        print(f"  First 3 rows:\n{dataframes[rs_id].head(3)}\n")
    else:
        print("  No records found for this record set.\n")


## 4. Exploratory Data Analysis (EDA)
Filter or transform numeric fields while referencing columns strictly by their field `@id`. For illustration, we'll choose the first record set with data and select a numeric field (by inspecting columns). Replace `<NUMERIC_FIELD_ID>` and `<GROUP_FIELD_ID>` with values from the field overview as needed.


In [ ]:
# Pick the first record set with data, and display its column @ids
main_rs_id = None
for rsid, df in dataframes.items():
    if df.shape[0] > 0:
        main_rs_id = rsid
        print(f"Using record set @id: {main_rs_id}")
        print(f"Available columns / field @ids: {df.columns.tolist()}")
        break

# Choose a numeric field's @id -- update as appropriate for this dataset
numeric_field_id = None
group_field_id = None
if main_rs_id:
    # Try to automatically select a numeric-looking field
    sampledf = dataframes[main_rs_id]
    for col in sampledf.columns:
        if pd.api.types.is_numeric_dtype(sampledf[col]):
            numeric_field_id = col
            break

    # Optionally: try to select potential grouping field
    for col in sampledf.columns:
        if sampledf[col].nunique() < 10 and sampledf[col].nunique() > 1:
            group_field_id = col
            break

    print(f"\nAnalysis will use numeric field @id: {numeric_field_id}")
    if group_field_id:
        print(f"Grouping by field @id: {group_field_id}")

    # Filter: keep records where value > 0 (if all > 0), else show a realistic filter
    if numeric_field_id:
        threshold = sampledf[numeric_field_id].mean() if sampledf[numeric_field_id].dtype in ['float', 'int'] else 0
        filtered_df = sampledf[sampledf[numeric_field_id] > threshold].copy()
        print(f"\nFiltered {len(filtered_df)}/{len(sampledf)} records with {numeric_field_id} > {threshold:.2f}.")
        
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized field preview:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping and aggregation
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nMean {numeric_field_id} by {group_field_id}:")
            print(grouped_df)


## 5. Visualization
Plot the (filtered) numeric field and, if available, group averages. All references are by field `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id:
    filtered_df = dataframes[main_rs_id]
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()


## 6. Conclusion
This notebook provided a reproducible pattern, using only `@id`s, for exploring and analyzing Croissant datasets via the `mlcroissant` API. Key steps included inspecting metadata, discovering available record sets/fields, loading data, basic EDA, and creating visualizations. For deeper analyses, you may join additional record sets (by `@id`) or apply domain-specific methods.
